(class2_exercise3)=
# Exercise 3. Classification on Text Metrics
An alternative way to represent text for classification could be via metrics like [POS tag](23-parts-of-speech-analysis=) proportions, sentence length or *readability*.  

This might give us greater insights into what distinguishes AI-generated from human-written text beyond word counts. Perhaps AI tends to use shorter sentences, making it easier for the classifier to detect. Perhaps not.

In [65]:
## CODE CHUNK REMOVED FOR USERS - HERE TO RELOAD DATA ##
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

## LOAD DATA ## 
# path of notebook
path = Path.cwd()

data_path = path.parents[1] / "resources" / "data" / "raid" / "train_none.csv"

raw_df = pd.read_csv(data_path)

## SUBSET DATA ##
df = raw_df[raw_df["model"].isin(["human", "cohere"])]

df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)

## SPLIT DATA ##
train_df, val_df= train_test_split(
                                                    df,
                                                    test_size=0.20,
                                                    random_state=42,
                                                    stratify=df["is_human"]
                                                    )

/var/folders/gg/gk923hkx2w3bw72pk2shplydry9j0b/T/ipykernel_3212/1245980808.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["is_human"] = df["model"].apply(lambda x: 1 if x == "human" else 0)


In [66]:
# THIS CHUNK IS HIDDEN FROM USERS - FUNCTIONS TO FIT AND EVALUATE CLASSIFIER FROM LAST NOTEBOOK ##
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

def clf_fit(X_train, y_train, random_state=42):
    """
    Fit a classifier on X and Y

    Could for example be X_train = X_train_bow and y_train = train_df['is_human']
    """
    clf = LogisticRegression(
    random_state=random_state,
    solver="liblinear",   # better for small/medium sparse datasets
    max_iter=1000,        # allow more iterations
    C=1.0,                # adjust if needed (smaller values = stronger regularization)
    )

    clf.fit(X_train, y_train)

    return clf

def clf_evaluate(clf_fitted, X_val, y_val):
    """
    Evaluate fitted classifier, extracting y_pred from X_val, and creating a classification report
    """
    y_pred = clf_fitted.predict(X_val)

    report = classification_report(y_val, y_pred)

    return report, y_pred 

## 3.1 Installing TextDescriptives
For this, we'll install the TextDescriptives package. This was actually developed by former cogsci legends {cite:t}`hansen_textdescriptives_2023`!!

In [67]:
%pip install textdescriptives==2.8.4


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


We also need to install *spacy* again!

In [68]:
%pip install spacy==3.8.7
!python -m spacy download en_core_web_sm


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 14.8 MB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: pip install --upgrade pip
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## 3.2 Extract Metrics

Import `textdescriptives` and `spacy`:

In [69]:
import textdescriptives as td
import spacy

Load our model:

In [70]:
nlp = spacy.load("en_core_web_sm")

We then add "text_descriptives" to our nlp pipeline (remember how our pipeline figure in last class had "ner", "tagger" - this is the same!)

In [71]:
nlp.add_pipe("textdescriptives/all")

Add our texts as `docs` (note: this is my train_df with `cohere` as defined in exercise 1):

In [103]:
train_docs = nlp.pipe(train_df["generation"], batch_size=16, n_process=-1) # use -1 to use all available cores. this way, if you have a larger machine, it will actually be more efficient.
val_docs = nlp.pipe(val_df["generation"], batch_size=16, n_process=-1)

### Your turn: Find Metrics to Extract
:::{admonition} HANDS-ON
:class: red
You can extract metrics as such: 
```python
train_metrics_df = td.extract_df(
                                    train_docs, 
                                    include_text=False, 
                                    metrics=["..", ".."] # this is pseudocode - check task!
                                    )
```
The additional `metrics` is not defined correctly in the snippet above. This is because I want you to choose which kinds of metrics you want (can be passed as a list).

**YOUR TASK**

1. Check the [docs](https://hlasse.github.io/TextDescriptives/extractors.html#textdescriptives.extractors.extract_df) and decide on which kinds of metrics you want to extract!
2. Extract the metrics for both `train_df` and `val_df`
3. Print one of the `metrics_df` to see how it looks!
:::

Solution:

In [104]:
train_metrics_df = td.extract_df(
                                    train_docs, 
                                    include_text=True, 
                                    metrics=["readability", "descriptive_stats"] # you can add more metrics here if you want!
                                    )

val_metrics_df = td.extract_df(
                                    val_docs, 
                                    include_text=True, 
                                    metrics=["readability", "descriptive_stats"] # you can add more metrics here if you want!
)

# might give a coherence warning, but I'm not extracting coherence here, so we'll ignore it!

/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/textdescriptives/components/coherence.py:44: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Span.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgements. This may happen if you're using one of the small models, e.g. `en_core_web_sm`, which don't ship with word vectors and only use context-sensitive tensors. You can always add your own word vectors, or use one of the larger models instead if available.
  similarities.append(sent.similarity(sents[i + order]))
/Users/au675000/Desktop/teaching/book.nosync/nlp-at-cogsci/.venv/lib/python3.12/site-packages/textdescriptives/components/coherence.py:44: UserWarning: [W007] The model you're using has no word vectors loaded, so the result of the Span.similarity method will be based on the tagger, parser and NER, which may not give useful similarity judgemen

You should have something like this (with your chosen metrics):

In [105]:
train_metrics_df.head()

,text,flesch_reading_ease,flesch_kincaid_grade,smog,gunning_fog,automated_readability_index,coleman_liau_index,lix,rix,token_length_mean,...,sentence_length_median,sentence_length_std,syllables_per_token_mean,syllables_per_token_median,syllables_per_token_std,n_tokens,n_unique_tokens,proportion_unique_tokens,n_characters,n_sentences
0,Ingredients:\n\n1/2 cup butter\n\n4 eggs\n\n1...,88.794773,3.411753,6.182691,5.204595,2.714765,5.068811,24.899600,1.500000,4.041958,...,9.0,7.262217,1.272727,1.0,0.504273,143,95,0.664336,603,14
1,VisDA is a new adaptation challenge that exte...,45.854481,13.446887,13.816670,15.883019,14.836792,11.817358,56.688679,8.000000,4.886792,...,24.5,6.344289,1.584906,1.0,0.878119,106,59,0.556604,533,4
2,The BBC has seen the official question for th...,60.920131,11.438692,12.815533,15.029070,13.377756,9.777395,49.200581,6.000000,4.537209,...,27.5,10.781205,1.402326,1.0,0.733576,430,158,0.367442,2024,16
3,Trio win right to sample AC/DC track\n\nJewis...,75.447199,8.785908,8.841846,11.383689,11.679804,9.117874,46.092072,5.294118,4.444444,...,25.0,6.919208,1.260870,1.0,0.568552,414,207,0.500000,1915,17
4,For the crust:\n1 cup all-purpose flour\n1/2 ...,92.614519,4.180802,4.851558,6.417112,4.444626,4.940000,31.336898,2.454545,3.852941,...,10.0,16.433688,1.164706,1.0,0.386449,170,84,0.494118,694,11


In [106]:
print(train_metrics_df.columns.tolist())

['text', 'flesch_reading_ease', 'flesch_kincaid_grade', 'smog', 'gunning_fog', 'automated_readability_index', 'coleman_liau_index', 'lix', 'rix', 'token_length_mean', 'token_length_median', 'token_length_std', 'sentence_length_mean', 'sentence_length_median', 'sentence_length_std', 'syllables_per_token_mean', 'syllables_per_token_median', 'syllables_per_token_std', 'n_tokens', 'n_unique_tokens', 'proportion_unique_tokens', 'n_characters', 'n_sentences']


## 3.3 Combine with Train and Val
Let's start by renaming the `text` column from our `metrics` dfs:

In [107]:
train_metrics_df = train_metrics_df.rename(columns={"text": "generation"})
val_metrics_df = val_metrics_df.rename(columns={"text": "generation"})

Let's use the `pd.merge` to combine the two dataframes on their shared column:

In [126]:
train_df_with_metrics = pd.merge(train_df, train_metrics_df.drop_duplicates(), on="generation") # for some reason, there were duplicate rows in metrics_df, so I dropped them first
val_df_with_metrics = pd.merge(val_df, val_metrics_df.drop_duplicates(), on="generation")

Check they are same length:

In [125]:
print(len(train_df_with_metrics), len(val_df_with_metrics))
print(len(train_df), len(val_df))

32090 8023
32090 8023


## 3.3 Remove NA's
Our logistic regression doesn't do well with NA's and we have a few of them from `textdescriptives`:

In [128]:
# print where NAN
print(train_df_with_metrics.isna().sum())

id                                 0
adv_source_id                      0
source_id                          0
model                              0
decoding                       10697
repetition_penalty             10697
attack                             0
domain                             0
title                              0
prompt                         10697
generation                         0
is_human                           0
flesch_reading_ease                1
flesch_kincaid_grade               1
smog                             497
gunning_fog                        1
automated_readability_index        1
coleman_liau_index                 1
lix                                1
rix                                1
token_length_mean                  1
token_length_median                1
token_length_std                   1
sentence_length_mean               0
sentence_length_median             0
sentence_length_std                0
syllables_per_token_mean           1
s

The NAs for `decoding` and `prompt` are expected, since they represent our `human` written text that did not have any technical parameters. We'll drop these. along with the metric `smog` to avoid removing to many rows due to them being `NA`:

In [132]:
# remove smog column to avoid dropping too many NA's (and do not drop NA's based on "decoding", "prompt"])
train_metrics_filtered = train_df_with_metrics.drop(columns=["smog", "decoding", "prompt"])
val_metrics_filtered = val_df_with_metrics.drop(columns=["smog", "decoding", "prompt"])

Now we can drop NA's

In [141]:
# now we can drop na's!
train_metrics_filtered = train_metrics_filtered.dropna()
val_metrics_filtered = val_metrics_filtered.dropna()

In [145]:
train_metrics_filtered

,id,adv_source_id,source_id,model,repetition_penalty,attack,domain,title,generation,is_human,...,sentence_length_median,sentence_length_std,syllables_per_token_mean,syllables_per_token_median,syllables_per_token_std,n_tokens,n_unique_tokens,proportion_unique_tokens,n_characters,n_sentences
0,9cecf20c-4709-49ca-be68-328ae56710a7,9cecf20c-4709-49ca-be68-328ae56710a7,840d4d53-58d9-438f-a656-de959cd6c449,cohere,no,none,recipes,Rci Flourless Chocolate Cake,Ingredients:\n\n1/2 cup butter\n\n4 eggs\n\n1...,0,...,9.0,7.262217,1.272727,1.0,0.504273,143,95,0.664336,603,14
1,26e7bd7e-dc69-4524-91e1-c4fb855f0819,26e7bd7e-dc69-4524-91e1-c4fb855f0819,849fb20c-767c-4f34-bca9-0b7d531380aa,cohere,no,none,abstracts,VisDA: The Visual Domain Adaptation Challenge,VisDA is a new adaptation challenge that exte...,0,...,24.5,6.344289,1.584906,1.0,0.878119,106,59,0.556604,533,4
2,c9b82c42-b62e-49f9-b85b-68c1a84ae4a1,c9b82c42-b62e-49f9-b85b-68c1a84ae4a1,97386e41-b31f-4f21-a2ab-aba2301c2596,cohere,no,none,news,EU referendum question unveiled,The BBC has seen the official question for th...,0,...,27.5,10.781205,1.402326,1.0,0.733576,430,158,0.367442,2024,16
3,eab0e600-37cd-46d2-90da-4435197158c2,eab0e600-37cd-46d2-90da-4435197158c2,da90a46d-613b-4ab3-8464-34474b3e894d,cohere,no,none,news,Beastie Boys win sampling battle,Trio win right to sample AC/DC track\n\nJewis...,0,...,25.0,6.919208,1.260870,1.0,0.568552,414,207,0.500000,1915,17
4,105b3a54-78bd-436d-bbc4-881529eedc8d,105b3a54-78bd-436d-bbc4-881529eedc8d,8cc7fdd8-3266-4caf-9ed5-174079370139,cohere,no,none,recipes,Cowboy Quiche 2,For the crust:\n1 cup all-purpose flour\n1/2 ...,0,...,10.0,16.433688,1.164706,1.0,0.386449,170,84,0.494118,694,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32084,87aa0a48-6473-4894-8f5b-e03ddc93dad7,87aa0a48-6473-4894-8f5b-e03ddc93dad7,ac8425d7-94d1-4e54-8fc2-9e9d49de0e67,cohere,no,none,reviews,12 Angry Men,This is movie making at its finest. 12 Angry ...,0,...,25.0,12.298996,1.349727,1.0,0.707561,183,114,0.622951,847,7
32086,43705dec-ab6b-4f53-8230-67e163f1f5dd,43705dec-ab6b-4f53-8230-67e163f1f5dd,c7ee9e8f-0b96-44cc-aad2-503d71c0b2ce,cohere,no,none,reddit,Difficulty exiting Reddit after being brought ...,Title: Difficulty exiting Reddit after being ...,0,...,15.5,6.905614,1.316327,1.0,0.616060,98,67,0.683673,424,8
32087,18cd2d51-865f-49a6-bd7a-a043ef5fb9a6,18cd2d51-865f-49a6-bd7a-a043ef5fb9a6,ddf6f89e-250d-456b-b598-8484a080f090,cohere,no,none,poetry,Lost And Given Over,Lost and given over\nTo a world that won't fo...,0,...,23.0,58.546456,1.191781,1.0,0.458034,219,101,0.461187,847,4
32088,89b43031-0aaf-4960-bb52-259722c442e8,89b43031-0aaf-4960-bb52-259722c442e8,30d7564e-03ff-40e4-b7b9-6fe9eb4b124a,cohere,no,none,abstracts,Some properties of the model of a superconduct...,We study a model of superconductivity which c...,0,...,17.0,4.166533,1.679012,1.0,1.120264,81,56,0.691358,444,5


## 3.3 Classify Metrics
You should now have a dataframes containing metrics for both `train` and `val` that are also filtered! We'll select the metrics:

In [142]:
metrics_to_use = ["flesch_reading_ease", "sentence_length_median", "n_unique_tokens", "n_tokens"] # you can change these to whatever you want!
X_train = train_metrics_filtered[metrics_to_use]
X_val = val_metrics_filtered[metrics_to_use]

### Your Turn: Use Your Pipeline Functions !

:::{admonition} HANDS-ON
:class: red
Now you'll reap the benefits of having defined functions in exercise 2:
- Select the metrics you are interested in classifying and create `X_train` and `X_val` (remember to copy the "remove NA" part if your metrics also contain a lot of NA's!)
- Use the functions from your pipeline that are relevant to fit and evaluate a classifier on your new *metric* features! 
- Compare the results to BOW/TF-IDF. Are they worse? Or better?
:::

#### Solution:
My solution, you might have another!

In [144]:
clf = clf_fit(X_train, train_metrics_filtered['is_human'])
report, y_pred = clf_evaluate(clf, X_val, val_metrics_filtered['is_human'])
print(report)

ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: 0

## 3.4 Explainable ML with SHAP Values
I promised you insights into *what* kind of features were important for our classification. We can do this by extracting `SHAP` values. I won't go in depth about what they are, but [read more about them here](https://shap.readthedocs.io/en/latest/example_notebooks/overviews/An%20introduction%20to%20explainable%20AI%20with%20Shapley%20values.html).

> Note: This only works properly if you choose metrics that don't heavily correlate. If they do so, it will be a coin toss whether it is metric 1 or metric 2 that shows up as the influential in our analysis. Perhaps choosing "n_tokens" and "n_sentences" is too risky.